In [ ]:
from fidelity_option_data_downloader import FidelityOptionDataDownloader
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')
ocd = FidelityOptionDataDownloader('chain', 'quotes', 'cookie.txt', logger=self.logger)
pd.set_option("display.max_columns", None)

In [ ]:
symlist = ['CRDO', 'MRVL', 'QQQ']#'IBM', 'TECH', 'NBIS', 'RKLB', 'MSTR', 'NOW', 'SMCI']

In [ ]:
with open('symbols.txt') as fo:
    symlist = [_.rstrip() for _ in fo]
print(len(symlist))

In [ ]:
symlist += ['CRDO']

In [ ]:
ocd.download_option_chain(symlist, batch_size=2)
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)

In [ ]:
df_raw = self.build_option_df(symlist)
dfcp = self.concat_put_call_options(df_raw)
self.check_data_age(df_raw)

In [ ]:
total_net_gex, strike_gex = self.do_gex(dfcp)
self.plot_total_gex(total_net_gex, top_n=6, W=total_net_gex.shape[0]*50)

In [ ]:
_df = total_net_gex
self.plot_gex_profiles(strike_gex, _df, R=0.2, W=2000)

In [ ]:
df_skew = self.calculate_iv_skew(dfcp[(dfcp.dte <= 180)])
px.bar(df_skew, x='expDt', y='25_delta_skew', color='symbol', barmode='group', height=500).show()
px.bar(df_skew, x='expDt', y='10_delta_skew', color='symbol', barmode='group', height=500).show()

In [ ]:
def estimate_spot_price(df, strike_col="strike", delta_col="Delta", dte_col="dte", type_col="type", types=["C", "P"]):
    spots_found = []
    foo = []
    # Process Calls and Puts separately if they exist
    for opt_type, target_delta in zip(types, [0.5, -0.5]):
        for dte in sorted(df[dte_col].unique()):
            sub_df = df[(df[type_col] == opt_type) & (df[dte_col] == dte)].copy()
            if len(sub_df) < 2:
                continue
            # Calculate absolute distance to the respective target delta
            sub_df["dist_to_atm"] = (sub_df[delta_col] - target_delta).abs()

            # Grab the two closest strikes that flank the target
            closest_two = sub_df.sort_values(by="dist_to_atm").head(2)

            S_1, S_2 = (
                closest_two[strike_col].iloc[0],
                closest_two[strike_col].iloc[1],
            )
            D_1, D_2 = (
                closest_two[delta_col].iloc[0],
                closest_two[delta_col].iloc[1],
            )

            # Avoid division by zero if two identical strikes somehow exist
            if D_2 != D_1:
                exact_spot = S_1 + (target_delta - D_1) * (S_2 - S_1) / (D_2 - D_1)
                spots_found.append(exact_spot)
                foo.append([opt_type, S_1.item(), S_2.item(), D_1.item(), D_2.item(), exact_spot.item()])

    # Return the average of Call and Put estimates, or whichever one was available
    if spots_found:
        print(foo)
        return np.mean(spots_found).item()
    else:
        raise ValueError(
            "Insufficient data to calculate interpolation for Calls or Puts."
        )

In [ ]:
_df = dfcp[(dfcp.symbol=='IBM')]

In [ ]:
estimate_spot_price(_df)

In [ ]:
_df.lastPrice

In [ ]:
_df[(_df.dte==4) & (_df.type=='P') & (_df.Delta >= -0.6) & (_df.Delta <= -0.4)]

In [ ]:
dfp = dfcp[dfcp.type == 'P'].drop(columns=['cluster', 'dte_cluster', 'overpaid', 'leverage', 'type'])

In [ ]:
dfp2 = self.compute_all_time_decay_metrics_for_symbols(dfp, ['QCOM'], ignore_no_bid=True, exclude_0dte=True, oi_lb=0)

In [ ]:
dfp2[(dfp2.strike <= 170) & (dfp2.dthProfit >= 50)].sort_values(by='dthr').head(20)